<a href="https://colab.research.google.com/github/AhmedMahmoud-123/FlyRank_AI/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
if not os.path.exists('FlyRank_AI'):
    !git clone -q https://github.com/AhmedMahmoud-123/FlyRank_AI.git
os.chdir('FlyRank_AI')
!python scripts/01_prepare_features.py

import pandas as pd
import numpy as np
df = pd.read_csv('data/processed/refresh_feature_vector.csv')
print(f'{len(df):,} rows')

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank_AI/FlyRank_AI/data/processed/refresh_feature_vector.csv
30,000 rows


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
traffic_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']
print(df[traffic_cols].describe(percentiles=[.5, .9, .99]))

# heavy-tail check: mean >> median is the signature
for col in traffic_cols:
    print(f"{col:20} mean={df[col].mean():>10.1f}   median={df[col].median():>8.1f}   max={df[col].max():>10.0f}")

       impressions_90d    clicks_90d  sessions_90d  ai_sessions_90d
count     30000.000000  30000.000000  30000.000000     30000.000000
mean       5200.366300     16.097333     37.066633         0.204500
std       16838.019547     75.076958    107.069131         1.363601
min           1.000000      0.000000      1.000000         0.000000
50%         731.000000      1.000000      7.000000         0.000000
90%       12136.400000     32.000000     88.000000         0.000000
99%       73505.830000    253.010000    451.010000         5.000000
max      517715.000000   4178.000000   4345.000000        64.000000
impressions_90d      mean=    5200.4   median=   731.0   max=    517715
clicks_90d           mean=      16.1   median=     1.0   max=      4178
sessions_90d         mean=      37.1   median=     7.0   max=      4345
ai_sessions_90d      mean=       0.2   median=     0.0   max=        64


**Distributions:** The traffic fields are strongly heavy-tailed. For example, `impressions_90d`
has a mean of 5,200.4 but a median of 731, with a maximum of 517,715. `clicks_90d` shows
the same pattern, with a mean of 16.1, median of 1, and maximum of 4,178. Because a small
number of pages account for very large values, the signal tests below use grouped medians
or rates rather than raw Pearson correlations.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# Signal test 1: does word count relate to impressions?

word_count_test = (
    df.groupby('word_count_tier')['impressions_90d']
      .agg(['median', 'count'])
      .sort_values('median', ascending=False)
)

print("Signal 1 — word count vs impressions")
display(word_count_test)


# Signal test 2: does freshness relate to trend?

freshness_test = pd.crosstab(
    df['freshness_tier'],
    df['trend_direction'],
    normalize='index'
).round(3)

print("\nSignal 2 — freshness vs trend")
display(freshness_test)


# Signal test 3: does competition relate to declining rate?

competition_test = (
    df.groupby('competition_level')['is_declining_label']
      .agg(['mean', 'count'])
      .sort_values('mean', ascending=False)
)

print("\nSignal 3 — competition vs declining rate")
display(competition_test)


# Overall declining rate

overall_declining_rate = df['is_declining_label'].mean()

print(
    f"\nOverall declining rate: "
    f"{overall_declining_rate:.3f} "
    f"({overall_declining_rate:.1%})"
)
# Small robustness check: word-count signal among pages with at least 250 impressions
robust_word_count_test = (
    df[df['impressions_90d'] >= 250]
      .groupby('word_count_tier')['impressions_90d']
      .agg(['median', 'count'])
      .sort_values('median', ascending=False)
)

print("\nRobustness check — word count vs impressions, pages with >=250 impressions")
print(robust_word_count_test)

Signal 1 — word count vs impressions


,median,count
word_count_tier,,
3500+,1340.0,6285
2000-3500,997.0,11263
unknown,878.0,7699
1000-2000,172.0,3780
<1000,4.0,973



Signal 2 — freshness vs trend


trend_direction,down,flat,new,stable,up
freshness_tier,,,,,
0-30,0.511,0.044,0.104,0.186,0.155
181+,0.471,0.092,0.144,0.138,0.155
31-90,0.589,0.006,0.040,0.149,0.217
91-180,0.611,0.026,0.008,0.229,0.126



Signal 3 — competition vs declining rate


,mean,count
competition_level,,
MEDIUM,0.566993,1836
LOW,0.563199,22896
HIGH,0.556057,2658
unknown,0.324904,2610



Overall declining rate: 0.542 (54.2%)

Robustness check — word count vs impressions, pages with >=250 impressions
                 median  count
word_count_tier               
3500+            4246.5   4174
2000-3500        2510.5   7762
unknown          1679.0   5708
1000-2000         976.0   1719
<1000             416.0     23


**SIGNAL 1 Verdict: CONFIRMED directionally.** Median impressions increase substantially across the
known word-count tiers, from 4 for pages under 1,000 words to 1,340 for pages above 3,500
words. This supports word count as a useful descriptive signal, but the relationship is
observational and does not show that increasing word count causes more impressions.

**SIGNAL 2 Verdict: MIXED.** Freshness does not show a monotonic relationship with trend direction.
The 31–90 day group has the highest observed `up` share (21.7%), while the 91–180 day group
has the highest `down` share (61.1%). The freshest 0–30 day group has only a 15.5% `up` share,
so freshness alone is not a reliable directional signal in this dataset.

**SIGNAL 3 Verdict: FALSE.** The observed declining rates are very similar across the known competition levels: HIGH (55.6%), MEDIUM (56.7%), and LOW (56.3%). Therefore, in this dataset, the observed decline rates do not show a meaningful directional difference across the known competition levels.

The `unknown` category is treated separately and is not used as evidence for this conclusion.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
striking = df[df['position_tier'].str.contains('striking', case=False, na=False)]
print(f"striking-distance pages: {len(striking):,}")
print(striking['is_declining_label'].mean(), "declining rate")
print(df['is_declining_label'].mean(), "overall declining rate")

striking-distance pages: 7,304
0.6095290251916758 declining rate
0.5420666666666667 overall declining rate


**Flag-linked test:** FlyRank's product flags treat striking-distance pages (position 11-20) as
high-priority recovery targets. The observed declining rate for striking-distance pages is
60.95% (n=7,304), compared with 54.21% across the full dataset (n=30,000).

**Verdict: OPPOSITE for the tested assumption.** In this dataset, striking-distance pages decline more often than the overall population, by about 6.7 percentage points. This is opposite to the assumption that these pages would be less likely to decline. However, the test does not evaluate recovery potential or business value, so it does not establish that the product flag itself is ineffective. For now, striking-distance status should be treated as a prioritization heuristic rather than evidence of lower decline risk.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The data supports using word count as a descriptive signal, but it does not justify treating content length as causal. Freshness shows a mixed relationship with trend, while competition level shows little difference in declining rates among known categories. Striking-distance pages have a higher observed declining rate than the overall dataset, so the flag should be treated as a prioritization heuristic rather than evidence about recovery potential.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card.